In [35]:
import pdfplumber
import pandas as pd

filepath = 'water rate data.pdf'

In [36]:
all_rows = []

def extract_pages_to_df(pdf_path, page_list):
    all_rows = []
    with pdfplumber.open(pdf_path) as pdf:
        for page_num in page_list:
            if 0 <= page_num < len(pdf.pages):
                page = pdf.pages[page_num]
                tables = page.extract_tables()
                for table in tables:
                    for row in table:
                        all_rows.append(row)
    return pd.DataFrame(all_rows)


### Create Separated Water/Wastewater Data csv

In [37]:
pages_separated = [20, 22, 24, 26, 28, 30]

df_separated = extract_pages_to_df(filepath, pages_separated)

separated_column_names = ['drop', 
                          'Utility Name',
                         'Water Minimum Charge',
                         'Water Minimum Gallons Included',
                         'Water Bill at 4,000 Gal (5.35Ccf)',
                         'Water Bill at 8,000 Gal (10.70 Ccf)',
                         'Wastewater Minimum Charge',
                         'Wastewater Minimum Gallons Included',
                         'Wastewater Bill at 4,000 Gal (5.35Ccf)',
                         'Wastewater Bill at 8,000 Gal (10.70 Ccf)',
                         'Res. Cap (Gal/Ccf)'
                         ]


df_separated.columns = separated_column_names
df_separated = df_separated.drop(columns=['drop'])
df_separated = df_separated[(df_separated['Water Minimum Charge'] != 'WATER') &
                            (~df_separated['Water Minimum Charge'].str.contains('Minimum', case=False))]

df_separated = df_separated.sort_values(by=['Utility Name'])


In [38]:
df_separated.head()

df_separated.to_csv('separatedratedata.csv', index=False)

### Create Combined Rates csv

In [39]:
pages_combined = [21, 23, 25, 27, 29, 31]

combined_column_names = ['County',
                         'Utility Name',
                         'Water Management District',
                         'Rates Effective Date',
                         'Unit',
                         'Minimum Charge',
                         'Bill at 4,000 Gal (5.35 Ccf)',
                         'Bill at 8,000 Gal (10.70 Ccf)',
                         'drop'
                         ]

df_combined = extract_pages_to_df(filepath, pages_combined)

df_combined.columns = combined_column_names

df_combined = df_combined.drop(columns=['drop'])

df_combined = df_combined[~(df_combined['Minimum Charge'].str.contains('COMBINED', case=False)) &
                          ~(df_combined['Minimum Charge'].str.contains('Minimum', case=False))]

df_combined['County'] = df_combined['County'].ffill()

df_combined = df_combined.sort_values(by=['Utility Name'])


In [40]:
df_combined.head()

df_combined.to_csv('combinedratedata.csv', index=False)